# Understand the RAG Dry Run

This notebook shows what the current dry run actually verifies. It does not call an API and it does not generate an answer. Instead, it walks one example through the pipeline up to the exact prompt messages that would be sent to a model.

## 1. Setup

The notebook imports the project package from `src/`, so it can run from the `notebooks/` directory without relying on a global install.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

PosixPath('/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project')

In [2]:
from rag_experiment.data.hotpotqa import load_hotpot_jsonl
from rag_experiment.retrieval.bm25 import BM25Retriever
from rag_experiment.generation.prompts import format_retrieved_context, get_prompt
from rag_experiment.runners.dry_run import run_config

DATA_PATH = PROJECT_ROOT / "data" / "hotpotqa_mini" / "sample.jsonl"
CONFIG_PATH = PROJECT_ROOT / "configs" / "hotpotqa_bm25_dry_run.json"
DATA_PATH, CONFIG_PATH

/opt/homebrew/Caskroom/miniconda/base/envs/LLM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(PosixPath('/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/data/hotpotqa_mini/sample.jsonl'),
 PosixPath('/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/configs/hotpotqa_bm25_dry_run.json'))

## 2. Load the Tiny HotpotQA-Style Dataset

For now this is a tiny synthetic fixture. It is useful for checking code flow, but it is not evidence for the final report.

In [3]:
examples = load_hotpot_jsonl(DATA_PATH)
print(f"examples: {len(examples)}")
print(f"total passages if flattened: {sum(len(example.passages()) for example in examples)}")
p = [example.passages() for example in examples]
print(p[0])
example = examples[0]
print(example)

examples: 5
total passages if flattened: 30
[Passage(id='mini_001::CN Tower::0', example_id='mini_001', title='CN Tower', sentence_index=0, text='The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.'), Passage(id='mini_001::CN Tower::1', example_id='mini_001', title='CN Tower', sentence_index=1, text='It was completed in 1976 and became a major landmark of the city.'), Passage(id='mini_001::University of Toronto::0', example_id='mini_001', title='University of Toronto', sentence_index=0, text='The University of Toronto is a public research university in Toronto, Ontario, Canada.'), Passage(id='mini_001::University of Toronto::1', example_id='mini_001', title='University of Toronto', sentence_index=1, text="It is located on the grounds that surround Queen's Park."), Passage(id='mini_001::McGill University::0', example_id='mini_001', title='McGill University', sentence_index=0, text='McGill University is a public research university in Montreal, Quebec, Can

In [4]:
example = examples[0]
print("id:", example.id)
print("question:", example.question)
print("gold answer:", example.answer)
print("type:", example.type)
print("level:", example.level)
print("supporting facts:", example.supporting_facts)

id: mini_001
question: Which university is located in the city where the CN Tower stands, the University of Toronto or McGill University?
gold answer: University of Toronto
type: comparison
level: easy
supporting facts: (('CN Tower', 0), ('University of Toronto', 0))


## 3. Inspect the Original Context

A HotpotQA-style example stores context as titled groups of sentences. Retrieval does not directly search this nested structure. First we turn the sentences into passage objects.

In [5]:
for title, sentences in example.context:
    print(f"\n## {title}")
    for index, sentence in enumerate(sentences):
        print(f"{index}: {sentence}")


## CN Tower
0: The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.
1: It was completed in 1976 and became a major landmark of the city.

## University of Toronto
0: The University of Toronto is a public research university in Toronto, Ontario, Canada.
1: It is located on the grounds that surround Queen's Park.

## McGill University
0: McGill University is a public research university in Montreal, Quebec, Canada.
1: It was established by royal charter in 1821.


## 4. Flatten Context into Passages

A passage is the smallest unit the retriever can return. In this first harness, each sentence becomes one passage with a stable `passage_id`.

In [6]:
passages = example.passages()
print(f"passages for this example: {len(passages)}")

for passage in passages:
    print(f"{passage.id} | {passage.text}")

passages for this example: 6
mini_001::CN Tower::0 | The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.
mini_001::CN Tower::1 | It was completed in 1976 and became a major landmark of the city.
mini_001::University of Toronto::0 | The University of Toronto is a public research university in Toronto, Ontario, Canada.
mini_001::University of Toronto::1 | It is located on the grounds that surround Queen's Park.
mini_001::McGill University::0 | McGill University is a public research university in Montreal, Quebec, Canada.
mini_001::McGill University::1 | It was established by royal charter in 1821.


## 5. Run LangChain BM25 Retrieval

The project does not hand-code BM25. `BM25Retriever` is our thin adapter around LangChain's BM25 retriever. The adapter returns project-owned records so later stages do not depend on LangChain internals.

In [7]:
top_k = 3
retriever = BM25Retriever(passages, top_k=top_k)
results = retriever.retrieve(example.question, top_k=top_k)

for result in results:
    passage = result.passage
    print(f"rank: {result.rank}")
    print(f"passage_id: {passage.id}")
    print(f"title: {passage.title}")
    print(f"sentence_index: {passage.sentence_index}")
    print(f"text: {passage.text}")
    print()

rank: 1
passage_id: mini_001::University of Toronto::1
title: University of Toronto
sentence_index: 1
text: It is located on the grounds that surround Queen's Park.

rank: 2
passage_id: mini_001::University of Toronto::0
title: University of Toronto
sentence_index: 0
text: The University of Toronto is a public research university in Toronto, Ontario, Canada.

rank: 3
passage_id: mini_001::McGill University::0
title: McGill University
sentence_index: 0
text: McGill University is a public research university in Montreal, Quebec, Canada.



## 6. Format Retrieved Context

This is the text block that will be inserted into the prompt. At this point we can already inspect whether retrieval gave the model enough evidence.

In [8]:
context = format_retrieved_context(results)
print(context)

[1] passage_id: mini_001::University of Toronto::1
title: University of Toronto
sentence_index: 1
text: It is located on the grounds that surround Queen's Park.

[2] passage_id: mini_001::University of Toronto::0
title: University of Toronto
sentence_index: 0
text: The University of Toronto is a public research university in Toronto, Ontario, Canada.

[3] passage_id: mini_001::McGill University::0
title: McGill University
sentence_index: 0
text: McGill University is a public research university in Montreal, Quebec, Canada.


## 7. Render the Prompt

The dry run stops here: it prepares the exact messages for a model call, but does not send them.

In [9]:
prompt = get_prompt("rag_qa_v1")
rendered_messages = prompt.render(question=example.question, context=context)

for message in rendered_messages:
    print("=" * 80)
    print(message["type"].upper())
    print("=" * 80)
    print(message["content"])
    print()

SYSTEM
You are a careful question-answering assistant. Answer using only the retrieved context. If the context is insufficient, say that the answer is not supported by the retrieved passages.

HUMAN
Question:
Which university is located in the city where the CN Tower stands, the University of Toronto or McGill University?

Retrieved context:
[1] passage_id: mini_001::University of Toronto::1
title: University of Toronto
sentence_index: 1
text: It is located on the grounds that surround Queen's Park.

[2] passage_id: mini_001::University of Toronto::0
title: University of Toronto
sentence_index: 0
text: The University of Toronto is a public research university in Toronto, Ontario, Canada.

[3] passage_id: mini_001::McGill University::0
title: McGill University
sentence_index: 0
text: McGill University is a public research university in Montreal, Quebec, Canada.

Return a concise answer. Include citations using passage ids in square brackets, for example [mini_001::Title::0].



## 8. What Dry Run Means

A dry run proves that the pipeline can prepare an inspectable model input. It does not prove the model answers correctly, because no model has been called.

In [10]:
dry_run_record = {
    "question": example.question,
    "gold_answer": example.answer,
    "retrieved_passage_count": len(results),
    "rendered_message_count": len(rendered_messages),
    "raw_model_answer": None,
    "parsed_answer": None,
}
print(json.dumps(dry_run_record, indent=2, ensure_ascii=False))

{
  "question": "Which university is located in the city where the CN Tower stands, the University of Toronto or McGill University?",
  "gold_answer": "University of Toronto",
  "retrieved_passage_count": 3,
  "rendered_message_count": 2,
  "raw_model_answer": null,
  "parsed_answer": null
}


## 9. Optional: Run the Existing Dry-Run Runner

This calls the same runner used from the command line. It writes a JSONL artifact under `outputs/`, with one record per example.

In [11]:
output_path = run_config(CONFIG_PATH)
print("wrote:", output_path)

with output_path.open("r", encoding="utf-8") as handle:
    first_record = json.loads(handle.readline())

print("first artifact keys:")
print(list(first_record.keys()))
print("error:", first_record["error"])
print("raw_model_answer:", first_record["raw_model_answer"])
print("retrieved passages:", len(first_record["retrieved_passages"]))

wrote: /Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/hotpotqa_mini_bm25_dry_run/results.jsonl
first artifact keys:
['run_name', 'created_at', 'dry_run', 'config', 'example', 'retrieved_passages', 'prompt', 'rendered_messages', 'raw_model_answer', 'parsed_answer', 'error']
error: None
raw_model_answer: None
retrieved passages: 3


## Current Takeaway

The dry run checks the plumbing. The next stage is a tiny generation run for one or two examples, where `raw_model_answer` becomes real text and we can inspect whether the model used the retrieved passages correctly.